In [2]:
import pandas as pd


In [6]:
df = pd.read_csv("climate_agg_country.csv")
df = df[(df['country'] == 'Kenya') | (df['country'] == 'Iran')]
df

,year,country,continent,climate_zone,mean_temp_C,mean_pr_mm_day,mean_spi,n_cells
105,1850,Iran,Asia,Temperate,17.9125,2.5027,-0.5168,4
115,1850,Kenya,Africa,Tropical,25.3926,0.8808,-0.3784,4
353,1851,Iran,Asia,Temperate,17.2869,0.8211,-0.1780,4
363,1851,Kenya,Africa,Tropical,25.6444,1.8367,-0.6983,4
601,1852,Iran,Asia,Temperate,17.9272,0.7077,-0.1648,4
...,...,...,...,...,...,...,...,...
40291,2012,Kenya,Africa,Tropical,26.0418,1.7038,-0.4874,4
40529,2013,Iran,Asia,Temperate,18.4883,0.9346,0.2464,4
40539,2013,Kenya,Africa,Tropical,26.4995,2.0398,-0.2984,4
40777,2014,Iran,Asia,Temperate,18.7245,1.0662,0.2279,4


In [11]:
import plotly.express as px

# 1. Filter the dataset down to just the two countries of interest
# (Ensure your DataFrame is named 'df')
df_filtered = df[df['country'].isin(['Iran', 'Kenya'])].sort_values(by='year')

# 2. Create the line graph
fig = px.line(
    df_filtered, 
    x="year", 
    y="mean_pr_mm_day", 
    color="country",
    hover_data=["climate_zone", "mean_temp_C", "mean_spi"],
    labels={
        "year": "Year",
        "mean_pr_mm_day": "Mean Precipitation (mm/day)",
        "country": "Country"
    },
    title="Historical Precipitation Trends: Iran vs. Kenya (1850–2014)"
)

# 3. Enhance styling for a cleaner look
fig.update_traces(line=dict(width=2))
fig.update_layout(
    template="plotly_white",
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# 4. Display the chart
fig.show()

In [12]:
import plotly.express as px

# 1. Filter the dataset down to just the two countries of interest
# (Ensure your DataFrame is named 'df')
df_filtered = df[df['country'].isin(['Iran', 'Kenya'])].sort_values(by='year')

# 2. Create the line graph for temperature
fig = px.line(
    df_filtered, 
    x="year", 
    y="mean_temp_C", 
    color="country",
    hover_data=["climate_zone", "mean_pr_mm_day", "mean_spi"],
    labels={
        "year": "Year",
        "mean_temp_C": "Mean Temperature (°C)",
        "country": "Country"
    },
    title="Historical Temperature Trends: Iran vs. Kenya (1850–2014)"
)

# 3. Enhance styling for a cleaner look
fig.update_traces(line=dict(width=2))
fig.update_layout(
    template="plotly_white",
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# 4. Display the chart
fig.show()

In [13]:
import plotly.express as px

# 1. Filter the dataset down to just the two countries of interest
df_filtered = df[df['country'].isin(['Iran', 'Kenya'])].sort_values(by='year')

# 2. Create the line graph for SPI
fig = px.line(
    df_filtered, 
    x="year", 
    y="mean_spi", 
    color="country",
    hover_data=["climate_zone", "mean_temp_C", "mean_pr_mm_day"],
    labels={
        "year": "Year",
        "mean_spi": "Standardized Precipitation Index (SPI)",
        "country": "Country"
    },
    title="Historical SPI Volatility: Iran vs. Kenya (1850–2014)"
)

# 3. Add a dashed baseline at SPI = 0 (Normal weather threshold)
fig.add_hline(
    y=0.0, 
    line_dash="dash", 
    line_color="gray", 
    annotation_text="Normal Baseline", 
    annotation_position="bottom right"
)

# 4. Enhance styling for clarity
fig.update_traces(line=dict(width=2))
fig.update_layout(
    template="plotly_white",
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray', range=[-3.5, 3.5]), # SPI scales usually stay between -3 and +3
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# 5. Display the chart
fig.show()

In [15]:
import pandas as pd

# 1. Filter for the two countries and pivot the table
df_pivot = df[df['country'].isin(['Iran', 'Kenya'])].pivot(index='year', columns='country', values='mean_spi')

# 2. CRITICAL STEP: Filter only for years where BOTH countries had negative SPIs (droughts)
drought_years = df_pivot[(df_pivot['Iran'] < 0) & (df_pivot['Kenya'] < 0)].copy()

# 3. Calculate the absolute difference between their SPIs for those specific drought years
drought_years['spi_difference'] = (drought_years['Iran'] - drought_years['Kenya']).abs()

# 4. Find the year with the minimum absolute difference among the drought years
closest_drought_year = drought_years['spi_difference'].idxmin()
min_diff = drought_years['spi_difference'].min()

# 5. Extract the exact SPI values for that year
iran_spi = drought_years.loc[closest_drought_year, 'Iran']
kenya_spi = drought_years.loc[closest_drought_year, 'Kenya']

print(f"The year where both countries were in a drought and closest in SPI is: {closest_drought_year}")
print(f"Iran SPI: {iran_spi:.5f} (Drought)")
print(f"Kenya SPI: {kenya_spi:.5f} (Drought)")
print(f"Absolute Difference: {min_diff:.5f}")

The year where both countries were in a drought and closest in SPI is: 1958
Iran SPI: -0.46960 (Drought)
Kenya SPI: -0.46390 (Drought)
Absolute Difference: 0.00570


In [ ]:
import plotly.express as px

fig = px.scatter(
    df, 
    x="mean_temp_C", 
    y="mean_pr_mm_day", 
    animation_frame="year", 
    animation_group="country",
    size="n_cells", 
    color="mean_spi", 
    hover_name="country",
    facet_col="continent", # Subplots per continent
    color_continuous_scale=px.colors.diverging.RdBu,
    color_continuous_midpoint=0,
    range_x=[df['mean_temp_C'].min()-1, df['mean_temp_C'].max()+1],
    range_y=[df['mean_pr_mm_day'].min()-0.5, df['mean_pr_mm_day'].max()+0.5],
    title="Climate Trajectories and SPI Fluctuations (1850-2014)"
)
fig.show()

In [8]:
import plotly.express as px

# Sort data to make the heatmap look structured
df_sorted = df.sort_values(by=["continent", "country", "year"])

fig = px.density_heatmap(
    df_sorted, 
    x="year", 
    y="country", 
    z="mean_spi",
    facet_row="continent",
    color_continuous_scale=px.colors.diverging.RdBu,
    color_continuous_midpoint=0,
    histfunc="avg",
    title="Historical Drought Footprints by Country and Continent",
    height=800
)
# Adjust y-axes to match the countries in each continent facet
fig.update_yaxes(matches=None) 
fig.show()

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Example template for a single country case study (e.g., Iran)
country_df = df[df['country'] == 'Iran'].sort_values('year')

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add SPI as bars
fig.add_trace(
    go.Bar(
        x=country_df['year'], 
        y=country_df['mean_spi'],
        marker_color=['#b2182b' if val < 0 else '#2166ac' for val in country_df['mean_spi']],
        name="Mean SPI"
    ),
    secondary_y=False,
)

# Add Temperature as a line
fig.add_trace(
    go.Scatter(
        x=country_df['year'], 
        y=country_df['mean_temp_C'], 
        line=dict(color='black', width=2),
        name="Mean Temp (°C)"
    ),
    secondary_y=True,
)

fig.update_layout(title_text="Iran Climate Case Study: Temperature Rise vs. SPI")
fig.update_yaxes(title_text="Standardized Precipitation Index (SPI)", secondary_y=False)
fig.update_yaxes(title_text="Temperature (°C)", secondary_y=True)
fig.show()

In [10]:
import plotly.express as px

fig = px.violin(
    df, 
    x="climate_zone", 
    y="mean_spi", 
    color="climate_zone",
    animation_frame="year",
    points="all", # Shows individual country dots inside the violin
    hover_data=["country"],
    box=True,
    title="Distribution of SPI Across Climate Zones Over Time"
)
fig.update_layout(yaxis=dict(range=[-3, 3])) # Keep SPI scale fixed
fig.show()